# Pairwise Sequence Alignment

**Tier 2 -- Core Bioinformatics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why** biologists align sequences and what an alignment represents biologically
2. Construct and interpret **dot plots** as a visual alignment tool
3. Describe substitution matrices (PAM, BLOSUM) and gap penalty schemes (linear, affine)
4. Implement the **Needleman-Wunsch** (global) algorithm from scratch with dynamic programming
5. Implement the **Smith-Waterman** (local) algorithm from scratch
6. Use BioPython's alignment tools for practical work
7. Interpret alignment statistics: scores, E-values, and percent identity

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: BioPython Essentials](../../Tier_2_Core_Bioinformatics/02_BioPython_Essentials/01_biopython_essentials.ipynb) | [Next: BLAST: Sequence Similarity Searching →](../../Tier_2_Core_Bioinformatics/04_BLAST_Searching/01_blast_searching.ipynb)

---

## 1. Why Align Sequences?

### The Biological Motivation

Over the course of evolution, an ancestral DNA or protein sequence accumulates **mutations**:

| Mutation type | Description | Alignment notation |
|---|---|---|
| **Substitution** | One residue is replaced by another | Mismatch column |
| **Insertion** | New residues are added | Gap (`-`) in the other sequence |
| **Deletion** | Residues are removed | Gap (`-`) in this sequence |

Because insertions in one lineage are indistinguishable from deletions in the other when we only have extant sequences, we collectively call them **indels**.

**A pairwise alignment is a hypothesis about which positions in two sequences are descended from the same ancestral position** (i.e., are *homologous*).

### Why This Matters

- **Homology detection**: Sequences that share significant similarity likely share a common ancestor
- **Function prediction**: Homologous proteins typically have related functions
- **Evolutionary analysis**: Alignments reveal which residues are conserved (functionally important)
- **Structure prediction**: Closely related sequences adopt similar 3D structures
- **Variant analysis**: Comparing a patient sequence to a reference reveals clinically relevant mutations

### A Concrete Example

Consider two derived sequences that evolved from the same ancestor. We do not know the ancestor -- we only see the modern sequences. Multiple alignments are possible, and the goal of alignment algorithms is to find the one that best explains the evolutionary relationship.

```
Possible alignment 1:              Possible alignment 2:

ACCGGTGGAACCGG-TAACACCCAC         ACCGGTGGAACCGGTAACACCCAC
ACCGGTA--ACCGGTTAACACCCAC         ACCGGTAACCGGTTAACACCCAC-
****** * ******.***********        *****  *  *  * *  * *** 

Score: 19                          Score: 8
```

Alignment 1 introduces two gaps but achieves much higher identity -- it likely better reflects what actually happened evolutionarily.

---

## 2. Dot Plots: Visual Alignment

Before diving into algorithms, let us visualize sequence similarity with **dot plots**. A dot plot is a simple 2D matrix where one sequence runs along the x-axis and the other along the y-axis. A dot is placed at position (i, j) when position i of one sequence matches position j of the other.

- **Diagonal lines** indicate regions of similarity
- **Broken diagonals** indicate insertions/deletions
- **Parallel diagonals** indicate repeated regions
- **Perpendicular diagonals** indicate palindromes / inverted repeats

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def dot_plot(seq1, seq2, title="Dot Plot"):
    """Create a basic dot plot between two sequences."""
    s1 = str(seq1)
    s2 = str(seq2)
    matrix = np.zeros((len(s2), len(s1)), dtype=int)
    for i, c2 in enumerate(s2):
        for j, c1 in enumerate(s1):
            if c1 == c2:
                matrix[i, j] = 1
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(matrix, cmap="Greys", aspect="auto", interpolation="none")
    ax.set_xlabel("Sequence 1")
    ax.set_ylabel("Sequence 2")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# Two related protein sequences (human and mouse hemoglobin alpha)
hba_human = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
hba_mouse = "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH"

dot_plot(hba_human, hba_mouse, "Human vs Mouse Hemoglobin Alpha (first 50 aa)")

The strong diagonal indicates high similarity between the two hemoglobin sequences. The few off-diagonal dots are noise from individual matching residues.

In [ ]:
def windowed_dot_plot(seq1, seq2, window=3, threshold=2, title="Windowed Dot Plot"):
    """Dot plot with a sliding window to reduce noise.
    
    A dot is placed at (i,j) if at least `threshold` of the `window`
    residues starting at (i,j) match.
    """
    s1, s2 = str(seq1), str(seq2)
    rows = len(s2) - window + 1
    cols = len(s1) - window + 1
    matrix = np.zeros((rows, cols), dtype=int)
    for i in range(rows):
        for j in range(cols):
            matches = sum(1 for k in range(window) if s2[i + k] == s1[j + k])
            if matches >= threshold:
                matrix[i, j] = 1
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(matrix, cmap="Greys", aspect="auto", interpolation="none")
    ax.set_xlabel("Sequence 1")
    ax.set_ylabel("Sequence 2")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


windowed_dot_plot(hba_human, hba_mouse, window=5, threshold=4,
                  title="Windowed Dot Plot (w=5, t=4)")

The windowed version dramatically reduces noise: only regions where several consecutive residues match are shown. This makes the alignment signal much clearer.

### Dot Plots Reveal Structural Features

Let us compare a sequence to itself to reveal internal repeats.

In [ ]:
# A sequence with an internal repeat
repeat_seq = "ACGTACGTACGTNNNNACGTACGTACGT"
windowed_dot_plot(repeat_seq, repeat_seq, window=4, threshold=4,
                  title="Self Dot Plot -- Internal Repeat")

The parallel diagonals in a self-comparison reveal repeated elements within the sequence.

---

## 3. Scoring Alignments

To decide which alignment is best, we need a **scoring system** with three components:

1. **Match/mismatch scores** -- how much to reward identical positions and penalise mismatches
2. **Substitution matrices** -- position-specific scores for amino acid pairs
3. **Gap penalties** -- the cost of introducing gaps (indels)

### 3.1 Simple Match/Mismatch Scoring

For DNA, a simple scheme often suffices:
- Match: +1 (or +2)
- Mismatch: -1
- Gap: -2

This works because there are only 4 nucleotides and transitions vs. transversions can be handled separately if needed.

### 3.2 Substitution Matrices for Proteins

For proteins, not all substitutions are equal. Replacing leucine (L) with isoleucine (I) -- both hydrophobic, branched-chain amino acids -- is far more likely in evolution than replacing leucine with aspartate (D), a negatively charged residue.

**Substitution matrices** encode the log-odds of observed substitution frequencies:

$$S(a, b) = \log_2 \frac{q_{ab}}{p_a \cdot p_b}$$

where $q_{ab}$ is the observed frequency of substitution and $p_a, p_b$ are background amino acid frequencies.

- **Positive score**: substitution observed more often than expected (conservative)
- **Negative score**: substitution observed less often than expected (disruptive)
- **Zero**: observed at expected frequency

### 3.3 PAM vs BLOSUM Matrices

| Feature | PAM | BLOSUM |
|---|---|---|
| **Full name** | Point Accepted Mutation | BLOcks SUbstitution Matrix |
| **Developed by** | Margaret Dayhoff (1978) | Henikoff & Henikoff (1992) |
| **Basis** | Closely related sequences, extrapolated | Conserved blocks across diverse proteins |
| **Numbering** | Higher number = more distant (PAM250) | Higher number = more similar (BLOSUM80) |
| **For close sequences** | PAM40 | BLOSUM80 |
| **For distant sequences** | PAM250 | BLOSUM45 |
| **Most commonly used** | PAM250 | **BLOSUM62** (default in BLAST) |

**Key insight**: PAM1 represents 1% divergence and is extrapolated by matrix exponentiation to higher evolutionary distances. BLOSUM matrices are directly computed from blocks of conserved sequences at the specified percent identity threshold.

In [ ]:
from Bio.Align import substitution_matrices

# Load BLOSUM62 -- the most widely used substitution matrix
blosum62 = substitution_matrices.load("BLOSUM62")

# Display a focused subset: hydrophobic amino acids
aa_subset = "AILMFYWV"
print("BLOSUM62 -- Hydrophobic amino acids:")
print("     ", "  ".join(f"{aa:>3}" for aa in aa_subset))
for aa1 in aa_subset:
    row = [f"{blosum62[aa1, aa2]:>3}" for aa2 in aa_subset]
    print(f"  {aa1}  ", "  ".join(row))

In [ ]:
# Compare a few informative pairs
pairs = [
    ("L", "I", "Both hydrophobic, branched-chain"),
    ("F", "Y", "Both aromatic"),
    ("D", "E", "Both negatively charged"),
    ("K", "R", "Both positively charged"),
    ("S", "T", "Both small, hydroxyl-containing"),
    ("L", "D", "Hydrophobic vs charged -- disruptive"),
    ("W", "G", "Large aromatic vs smallest amino acid"),
]

print(f"{'Pair':<8} {'Score':>6}  Reason")
print("-" * 55)
for a, b, reason in pairs:
    score = blosum62[a, b]
    print(f"{a} <-> {b}  {score:>+4}    {reason}")

In [ ]:
# Visualize the full BLOSUM62 matrix as a heatmap
standard_aa = "ARNDCQEGHILKMFPSTWYV"
matrix_data = np.zeros((20, 20))
for i, a in enumerate(standard_aa):
    for j, b in enumerate(standard_aa):
        matrix_data[i, j] = blosum62[a, b]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matrix_data, cmap="RdBu_r", vmin=-4, vmax=11)
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(list(standard_aa))
ax.set_yticklabels(list(standard_aa))
ax.set_title("BLOSUM62 Substitution Matrix")
plt.colorbar(im, label="Score")
plt.tight_layout()
plt.show()

**Observations from the heatmap:**
- The diagonal (self-substitutions) is always positive: identical residues are rewarded
- Tryptophan (W) has the highest self-score (11) -- it is the rarest amino acid, so seeing it conserved is very informative
- Clusters of positive off-diagonal scores reflect chemical similarity groups (e.g., I/L/M/V, D/E, K/R)

### 3.4 Gap Penalties

Gaps represent insertions or deletions (indels). Two common penalty schemes:

**Linear gap penalty:**
$$\text{gap\_cost}(k) = -d \times k$$

where $d$ is the penalty per gap position and $k$ is the gap length. Every gap position costs the same.

**Affine gap penalty:**
$$\text{gap\_cost}(k) = -d - (k - 1) \times e$$

where $d$ is the **gap opening** penalty and $e$ is the **gap extension** penalty ($e < d$).

| Scheme | Advantages | Typical values |
|---|---|---|
| Linear | Simpler, fewer parameters | $d = 8$ |
| Affine | More biologically realistic | $d = 10$, $e = 0.5$ |

**Why affine?** In biology, a single mutational event can insert or delete several residues at once (e.g., replication slippage). A single long gap is biologically more plausible than many scattered single-residue gaps. Affine penalties encourage this by making gap *opening* expensive but gap *extension* cheap.

In [ ]:
# Compare linear vs affine gap costs
gap_lengths = np.arange(1, 16)

d_linear = 8
linear_cost = d_linear * gap_lengths

d_open, e_extend = 10, 0.5
affine_cost = d_open + (gap_lengths - 1) * e_extend

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gap_lengths, linear_cost, "o-", label=f"Linear (d={d_linear})")
ax.plot(gap_lengths, affine_cost, "s-", label=f"Affine (d={d_open}, e={e_extend})")
ax.set_xlabel("Gap length (residues)")
ax.set_ylabel("Total penalty")
ax.set_title("Linear vs Affine Gap Penalty")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With affine penalties, short gaps are penalized MORE than linear,")
print("but long gaps are penalized LESS -- favouring fewer, longer gaps.")

---

## 4. Global Alignment: The Needleman-Wunsch Algorithm

Published in 1970 by Saul Needleman and Christian Wunsch, this was one of the first applications of **dynamic programming** to biology.

**Global alignment** aligns two sequences end-to-end in their entirety. It is appropriate when the sequences are of similar length and suspected to be homologous over their full length.

### Algorithm Overview

Given sequences $X$ of length $m$ and $Y$ of length $n$:

1. **Create** a score matrix $F$ of size $(n+1) \times (m+1)$ and a traceback matrix $T$
2. **Initialize** the first row and column with cumulative gap penalties
3. **Fill** each cell using the recurrence relation
4. **Traceback** from the bottom-right corner to recover the alignment

### Recurrence Relation

$$F(i, j) = \max \begin{cases} F(i-1, j-1) + s(x_j, y_i) & \text{(match/mismatch)} \\ F(i-1, j) - d & \text{(gap in sequence X)} \\ F(i, j-1) - d & \text{(gap in sequence Y)} \end{cases}$$

where $s(x_j, y_i)$ is the substitution score and $d$ is the gap penalty.

### Time and Space Complexity

- **Time**: $O(m \times n)$ -- we fill every cell in the matrix
- **Space**: $O(m \times n)$ -- we store the full matrix for traceback

In [ ]:
def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    """Needleman-Wunsch global alignment with linear gap penalty.
    
    Parameters
    ----------
    seq1, seq2 : str
        Sequences to align.
    match : int
        Score for matching positions.
    mismatch : int
        Score for mismatching positions.
    gap : int
        Penalty for each gap position (should be negative).
    
    Returns
    -------
    aligned1, aligned2 : str
        The aligned sequences (with gaps as '-').
    score : int
        The optimal alignment score.
    F : np.ndarray
        The score matrix (for visualization).
    """
    m, n = len(seq1), len(seq2)
    
    # Step 1: Initialize score matrix and traceback matrix
    F = np.zeros((n + 1, m + 1), dtype=int)
    T = np.zeros((n + 1, m + 1), dtype=int)  # 0=diag, 1=up, 2=left
    
    # Step 2: Fill first row and column with gap penalties
    for i in range(1, n + 1):
        F[i, 0] = gap * i
        T[i, 0] = 1  # came from above
    for j in range(1, m + 1):
        F[0, j] = gap * j
        T[0, j] = 2  # came from left
    
    # Step 3: Fill the rest of the matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Score for aligning seq2[i-1] with seq1[j-1]
            s = match if seq1[j - 1] == seq2[i - 1] else mismatch
            
            diag = F[i - 1, j - 1] + s    # match/mismatch
            up   = F[i - 1, j] + gap       # gap in seq1
            left = F[i, j - 1] + gap       # gap in seq2
            
            F[i, j] = max(diag, up, left)
            
            if F[i, j] == diag:
                T[i, j] = 0
            elif F[i, j] == up:
                T[i, j] = 1
            else:
                T[i, j] = 2
    
    # Step 4: Traceback from bottom-right corner
    aligned1, aligned2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and T[i, j] == 0:
            aligned1.append(seq1[j - 1])
            aligned2.append(seq2[i - 1])
            i -= 1
            j -= 1
        elif i > 0 and T[i, j] == 1:
            aligned1.append("-")
            aligned2.append(seq2[i - 1])
            i -= 1
        else:
            aligned1.append(seq1[j - 1])
            aligned2.append("-")
            j -= 1
    
    aligned1 = "".join(reversed(aligned1))
    aligned2 = "".join(reversed(aligned2))
    score = F[n, m]
    
    return aligned1, aligned2, score, F


print("Needleman-Wunsch function defined.")

In [ ]:
# Test with short sequences
seq_a = "HEAGAWGHEE"
seq_b = "PAWHEAE"

a1, a2, score, F = needleman_wunsch(seq_a, seq_b, match=2, mismatch=-1, gap=-2)

print(f"Sequence 1: {seq_a}")
print(f"Sequence 2: {seq_b}")
print(f"\nGlobal alignment (Needleman-Wunsch):")
print(f"  {a1}")
# Build match line
match_line = "".join("|" if c1 == c2 else " " for c1, c2 in zip(a1, a2))
print(f"  {match_line}")
print(f"  {a2}")
print(f"\nScore: {score}")

### Visualizing the Dynamic Programming Matrix

Understanding the DP matrix is key to understanding the algorithm. Let us visualize it.

In [ ]:
def visualize_dp_matrix(F, seq1, seq2, title="Dynamic Programming Matrix", path=None):
    """Visualize the score matrix as a heatmap with cell values."""
    fig, ax = plt.subplots(figsize=(max(8, len(seq1) * 0.8), max(6, len(seq2) * 0.8)))
    im = ax.imshow(F, cmap="YlOrRd", aspect="auto")
    
    # Add text annotations
    for i in range(F.shape[0]):
        for j in range(F.shape[1]):
            ax.text(j, i, str(F[i, j]), ha="center", va="center", fontsize=9)
    
    # Labels
    col_labels = ["-"] + list(seq1)
    row_labels = ["-"] + list(seq2)
    ax.set_xticks(range(len(col_labels)))
    ax.set_yticks(range(len(row_labels)))
    ax.set_xticklabels(col_labels)
    ax.set_yticklabels(row_labels)
    ax.xaxis.tick_top()
    ax.set_title(title, pad=20)
    plt.colorbar(im, label="Score")
    plt.tight_layout()
    plt.show()


visualize_dp_matrix(F, seq_a, seq_b, "Needleman-Wunsch Score Matrix")

The optimal alignment score is found in the **bottom-right cell** of the matrix. The traceback follows the path of cells that produced this optimal score.

### Step-by-Step Walkthrough

Let us trace through the algorithm manually with DNA sequences.

In [ ]:
# Small DNA example for manual walkthrough
dna1 = "ACGT"
dna2 = "AGT"

a1, a2, score, F = needleman_wunsch(dna1, dna2, match=1, mismatch=-1, gap=-2)

print("Step-by-step Needleman-Wunsch:")
print(f"  Seq1: {dna1}")
print(f"  Seq2: {dna2}")
print(f"  Match=+1, Mismatch=-1, Gap=-2")
print()
print("Score matrix F:")
print()

# Print the matrix with labels
header = "       -    " + "    ".join(dna1)
print(header)
row_labels = ["-"] + list(dna2)
for i, label in enumerate(row_labels):
    row = "    ".join(f"{F[i, j]:>2}" for j in range(F.shape[1]))
    print(f"  {label}  {row}")

print(f"\nOptimal alignment:")
print(f"  {a1}")
match_line = "".join("|" if c1 == c2 else " " for c1, c2 in zip(a1, a2))
print(f"  {match_line}")
print(f"  {a2}")
print(f"  Score: {score}")

---

## 5. Local Alignment: The Smith-Waterman Algorithm

Published in 1981 by Temple Smith and Michael Waterman, this algorithm finds the **best local alignment** -- the highest-scoring sub-region of similarity between two sequences.

### Three Key Differences from Needleman-Wunsch

| Feature | Needleman-Wunsch (Global) | Smith-Waterman (Local) |
|---|---|---|
| **Initialization** | First row/col = cumulative gap penalties | First row/col = 0 |
| **Recurrence** | $\max(\text{diag}, \text{up}, \text{left})$ | $\max(0, \text{diag}, \text{up}, \text{left})$ |
| **Traceback start** | Bottom-right cell | Cell with the **maximum score** anywhere |
| **Traceback end** | Top-left corner (0,0) | First cell with score 0 |

The critical addition of **0** in the recurrence means that no cell can ever go negative -- if the best option is to start fresh, the algorithm does so. This is what allows it to find local (partial) alignments.

In [ ]:
def smith_waterman(seq1, seq2, match=2, mismatch=-1, gap=-1):
    """Smith-Waterman local alignment with linear gap penalty.
    
    Returns
    -------
    aligned1, aligned2 : str
        The locally aligned sub-sequences (with gaps as '-').
    score : int
        The optimal local alignment score.
    F : np.ndarray
        The score matrix.
    """
    m, n = len(seq1), len(seq2)
    
    # Initialize -- all zeros (no cumulative gap penalties)
    F = np.zeros((n + 1, m + 1), dtype=int)
    T = np.zeros((n + 1, m + 1), dtype=int)  # 0=stop, 1=diag, 2=up, 3=left
    
    max_score = 0
    max_i, max_j = 0, 0
    
    # Fill the matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if seq1[j - 1] == seq2[i - 1] else mismatch
            
            diag = F[i - 1, j - 1] + s
            up   = F[i - 1, j] + gap
            left = F[i, j - 1] + gap
            
            F[i, j] = max(0, diag, up, left)  # KEY DIFFERENCE: zero floor
            
            if F[i, j] == 0:
                T[i, j] = 0  # stop
            elif F[i, j] == diag:
                T[i, j] = 1
            elif F[i, j] == up:
                T[i, j] = 2
            else:
                T[i, j] = 3
            
            # Track the maximum score position
            if F[i, j] > max_score:
                max_score = F[i, j]
                max_i, max_j = i, j
    
    # Traceback from the maximum score cell
    aligned1, aligned2 = [], []
    i, j = max_i, max_j
    while T[i, j] != 0 and i > 0 and j > 0:
        if T[i, j] == 1:  # diagonal
            aligned1.append(seq1[j - 1])
            aligned2.append(seq2[i - 1])
            i -= 1
            j -= 1
        elif T[i, j] == 2:  # up
            aligned1.append("-")
            aligned2.append(seq2[i - 1])
            i -= 1
        else:  # left
            aligned1.append(seq1[j - 1])
            aligned2.append("-")
            j -= 1
    
    aligned1 = "".join(reversed(aligned1))
    aligned2 = "".join(reversed(aligned2))
    
    return aligned1, aligned2, max_score, F


print("Smith-Waterman function defined.")

In [ ]:
# Demonstrate local alignment -- find a shared domain within longer sequences
long1 = "XXXXXXXXHEAGAWGHEEXXXXXXXX"
long2 = "YYYYPAWHEAEYYY"

a1, a2, score, F_sw = smith_waterman(long1, long2, match=2, mismatch=-1, gap=-1)

print(f"Sequence 1: {long1}")
print(f"Sequence 2: {long2}")
print(f"\nLocal alignment (Smith-Waterman):")
print(f"  {a1}")
match_line = "".join("|" if c1 == c2 else " " for c1, c2 in zip(a1, a2))
print(f"  {match_line}")
print(f"  {a2}")
print(f"\nScore: {score}")
print("\nNote: The algorithm ignores the non-matching X and Y flanks.")

In [ ]:
visualize_dp_matrix(F_sw, long1, long2, "Smith-Waterman Score Matrix")

### Comparing Global vs Local Alignment

Let us align the same pair of sequences with both algorithms to see the difference.

In [ ]:
s1 = "ACCTAAGG"
s2 = "GGCTCAATCA"

# Global alignment
ga1, ga2, gs, _ = needleman_wunsch(s1, s2, match=2, mismatch=-1, gap=-2)
print("GLOBAL alignment (Needleman-Wunsch):")
print(f"  {ga1}")
print(f"  {''.join('|' if a == b else ' ' for a, b in zip(ga1, ga2))}")
print(f"  {ga2}")
print(f"  Score: {gs}\n")

# Local alignment
la1, la2, ls, _ = smith_waterman(s1, s2, match=2, mismatch=-1, gap=-2)
print("LOCAL alignment (Smith-Waterman):")
print(f"  {la1}")
print(f"  {''.join('|' if a == b else ' ' for a, b in zip(la1, la2))}")
print(f"  {la2}")
print(f"  Score: {ls}")

print("\nGlobal alignment forces entire sequences to be aligned.")
print("Local alignment finds the best matching sub-region.")

### When to Use Which?

| Scenario | Algorithm | Why |
|---|---|---|
| Two full-length orthologous proteins | **Global** (NW) | Entire sequences are expected to align |
| Finding a domain within a larger protein | **Local** (SW) | Only a sub-region matches |
| Comparing a short read to a reference | **Local** (SW) or semi-global | Read is a fragment |
| Two sequences of very different length | **Local** (SW) | Global would force too many gaps |
| Database search (BLAST) | **Local** (SW) | Targets may be partially similar |

---

## 6. From-Scratch Implementation with Substitution Matrix Support

Our implementations above used simple match/mismatch scoring. Let us now build a version that accepts a substitution matrix (like BLOSUM62) and affine gap penalties.

In [ ]:
def nw_affine(seq1, seq2, sub_matrix, gap_open=-10, gap_extend=-0.5):
    """Needleman-Wunsch with substitution matrix and affine gap penalties."""
    m, n = len(seq1), len(seq2)
    
    # Three matrices for affine gaps
    # M: best score ending in a match/mismatch
    # X: best score ending in a gap in seq1 (insertion in seq2)
    # Y: best score ending in a gap in seq2 (insertion in seq1)
    NEG_INF = float("-inf")
    M = np.full((n + 1, m + 1), NEG_INF)
    X = np.full((n + 1, m + 1), NEG_INF)
    Y = np.full((n + 1, m + 1), NEG_INF)
    T = np.zeros((n + 1, m + 1), dtype=int)  # traceback: 0=M, 1=X, 2=Y
    
    # Initialization
    M[0, 0] = 0
    for i in range(1, n + 1):
        X[i, 0] = gap_open + (i - 1) * gap_extend
    for j in range(1, m + 1):
        Y[0, j] = gap_open + (j - 1) * gap_extend
    
    # Fill
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = sub_matrix[seq1[j - 1], seq2[i - 1]]
            
            M[i, j] = max(M[i-1, j-1], X[i-1, j-1], Y[i-1, j-1]) + s
            
            X[i, j] = max(
                M[i-1, j] + gap_open,
                X[i-1, j] + gap_extend
            )
            
            Y[i, j] = max(
                M[i, j-1] + gap_open,
                Y[i, j-1] + gap_extend
            )
    
    # Best score at the end
    final_scores = [M[n, m], X[n, m], Y[n, m]]
    best_matrix = np.argmax(final_scores)
    score = final_scores[best_matrix]
    
    # Simplified traceback (using the combined best score per cell)
    F_combined = np.maximum(np.maximum(M, X), Y)
    
    # Traceback
    aligned1, aligned2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = sub_matrix[seq1[j-1], seq2[i-1]]
            if i > 0 and j > 0 and F_combined[i, j] == F_combined[i-1, j-1] + s:
                aligned1.append(seq1[j-1])
                aligned2.append(seq2[i-1])
                i -= 1
                j -= 1
                continue
        if i > 0:
            gap_score_up = F_combined[i-1, j] + (gap_open if (i == n and j == m) or True else gap_extend)
            if F_combined[i, j] - F_combined[i-1, j] <= 0:
                aligned1.append("-")
                aligned2.append(seq2[i-1])
                i -= 1
                continue
        if j > 0:
            aligned1.append(seq1[j-1])
            aligned2.append("-")
            j -= 1
            continue
        # fallback
        if i > 0:
            aligned1.append("-")
            aligned2.append(seq2[i-1])
            i -= 1
        elif j > 0:
            aligned1.append(seq1[j-1])
            aligned2.append("-")
            j -= 1
    
    return "".join(reversed(aligned1)), "".join(reversed(aligned2)), score


# Test with protein sequences and BLOSUM62
prot1 = "HEAGAWGHEE"
prot2 = "PAWHEAE"

a1, a2, score = nw_affine(prot1, prot2, blosum62, gap_open=-10, gap_extend=-0.5)
print(f"Affine NW alignment with BLOSUM62:")
print(f"  {a1}")
print(f"  {''.join('|' if c1 == c2 and c1 != '-' else ' ' for c1, c2 in zip(a1, a2))}")
print(f"  {a2}")
print(f"  Score: {score}")

---

## 7. Using BioPython for Pairwise Alignment

In practice, you will use optimized libraries rather than writing alignment algorithms from scratch. BioPython provides the `Bio.Align.PairwiseAligner` class, which supports both global and local alignment, arbitrary substitution matrices, and affine gap penalties.

### The Modern API: `Bio.Align.PairwiseAligner`

In [ ]:
from Bio.Align import PairwiseAligner

# Create an aligner with default settings
aligner = PairwiseAligner()
print("Default aligner settings:")
print(f"  Mode: {aligner.mode}")
print(f"  Match score: {aligner.match_score}")
print(f"  Mismatch score: {aligner.mismatch_score}")
print(f"  Open gap score: {aligner.open_gap_score}")
print(f"  Extend gap score: {aligner.extend_gap_score}")

In [ ]:
# Global alignment with BLOSUM62 and affine gap penalties
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = blosum62
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

# Human and mouse hemoglobin alpha (first 50 residues)
hba_h = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
hba_m = "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH"

alignments = aligner.align(hba_h, hba_m)
best = alignments[0]

print(f"Number of optimal alignments: {len(alignments)}")
print(f"Score: {best.score}")
print()
print(best)

In [ ]:
# Local alignment
aligner_local = PairwiseAligner()
aligner_local.mode = "local"
aligner_local.substitution_matrix = blosum62
aligner_local.open_gap_score = -10
aligner_local.extend_gap_score = -0.5

# Find a short motif within a longer sequence
query = "AWGKVGAHAG"
target = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"

local_alns = aligner_local.align(target, query)
print(f"Score: {local_alns[0].score}")
print()
print(local_alns[0])

### Comparing Different Substitution Matrices

In [ ]:
# How does the choice of matrix affect alignment?
# KRAS protein: human vs. a more distant ortholog
kras_human = "MTEYKLVVVGAVGVGKSALTIQLIQNHFVDEYDPT"
kras_distant = "MTEYKIVVLGAVGIGKSALTIQFIQSHFVDQYDPT"

matrices_to_test = ["BLOSUM45", "BLOSUM62", "BLOSUM80"]

print(f"Human KRAS:   {kras_human}")
print(f"Distant:      {kras_distant}")
print()

for mat_name in matrices_to_test:
    mat = substitution_matrices.load(mat_name)
    a = PairwiseAligner()
    a.mode = "global"
    a.substitution_matrix = mat
    a.open_gap_score = -10
    a.extend_gap_score = -0.5
    
    result = a.align(kras_human, kras_distant)
    print(f"{mat_name}: score = {result[0].score:.1f}, "
          f"{len(result)} optimal alignment(s)")

---

## 8. Alignment Statistics and Interpretation

Raw alignment scores are not directly comparable across different searches because they depend on sequence lengths, composition, and scoring parameters. We need normalized statistics.

### 8.1 Percent Identity and Similarity

| Metric | Definition |
|---|---|
| **Percent identity** | Fraction of aligned positions with identical residues |
| **Percent similarity** | Fraction with identical OR conservative substitutions (positive BLOSUM62 score) |
| **Gaps** | Number of gap positions in the alignment |
| **Alignment length** | Total columns in the alignment |

### The Twilight Zone

For proteins:
- **>30% identity** over a significant length: almost certainly homologous
- **20--30% identity**: the "twilight zone" -- may or may not be homologous
- **<20% identity**: the "midnight zone" -- cannot distinguish from random

In [ ]:
def alignment_statistics(aligned_seq1, aligned_seq2, sub_matrix=None):
    """Calculate detailed alignment statistics."""
    if sub_matrix is None:
        sub_matrix = substitution_matrices.load("BLOSUM62")
    
    identities = 0
    similarities = 0
    gaps = 0
    mismatches = 0
    
    for a, b in zip(str(aligned_seq1), str(aligned_seq2)):
        if a == "-" or b == "-":
            gaps += 1
        elif a == b:
            identities += 1
            similarities += 1
        else:
            mismatches += 1
            try:
                if sub_matrix[a, b] > 0:
                    similarities += 1
            except (KeyError, IndexError):
                pass
    
    total_length = len(str(aligned_seq1))
    aligned_positions = total_length - gaps
    
    stats = {
        "alignment_length": total_length,
        "aligned_positions": aligned_positions,
        "identities": identities,
        "identity_pct": 100 * identities / aligned_positions if aligned_positions else 0,
        "similarities": similarities,
        "similarity_pct": 100 * similarities / aligned_positions if aligned_positions else 0,
        "mismatches": mismatches,
        "gaps": gaps,
        "gap_pct": 100 * gaps / total_length if total_length else 0,
    }
    return stats


# Compute statistics for the hemoglobin alignment
stats = alignment_statistics(hba_h, hba_m)
print("Hemoglobin alpha alignment statistics:")
print(f"  Alignment length:  {stats['alignment_length']}")
print(f"  Identities:        {stats['identities']}/{stats['aligned_positions']} "
      f"({stats['identity_pct']:.1f}%)")
print(f"  Similarities:      {stats['similarities']}/{stats['aligned_positions']} "
      f"({stats['similarity_pct']:.1f}%)")
print(f"  Gaps:              {stats['gaps']} ({stats['gap_pct']:.1f}%)")

### 8.2 Bit Scores and E-values

When searching databases (e.g., with BLAST), two statistics are critical:

**Bit score** normalizes the raw alignment score to be comparable across different searches:

$$S' = \frac{\lambda S - \ln K}{\ln 2}$$

where $S$ is the raw score, and $\lambda$, $K$ are Karlin-Altschul statistical parameters that depend on the scoring scheme and sequence composition.

**E-value** (Expect value) estimates how many alignments with this score we would expect to find by chance in a database of a given size:

$$E = K \cdot m \cdot n \cdot e^{-\lambda S}$$

or equivalently: $E = m \cdot n \cdot 2^{-S'}$

| E-value | Interpretation |
|---|---|
| $< 10^{-50}$ | Virtually identical sequences |
| $10^{-50}$ to $10^{-10}$ | Clear homologs, same family |
| $10^{-10}$ to $10^{-5}$ | Probable distant homologs |
| $10^{-5}$ to $0.01$ | Possible homology -- investigate further |
| $> 1$ | Likely not significant |

**Key points:**
- E-value depends on **database size** -- the same alignment gets a larger (worse) E-value in a larger database
- Bit score is **database-independent** -- comparable across searches
- Always report E-values with the database used and its version

In [ ]:
# Demonstrate how E-value scales with database size
import math

bit_score = 150  # fixed alignment quality
query_length = 300

db_sizes = {
    "SwissProt (~500K seqs)": 2e8,
    "UniProt/TrEMBL (~250M seqs)": 8e10,
    "nr (~500M seqs)": 2e11,
}

print(f"Same alignment with bit score = {bit_score}:")
print(f"{'Database':<35} {'Effective size':>15} {'E-value':>15}")
print("-" * 68)
for name, n in db_sizes.items():
    e_value = query_length * n * 2**(-bit_score)
    print(f"{name:<35} {n:>15.1e} {e_value:>15.2e}")

print("\nThe same alignment is more 'significant' in a smaller database!")

### 8.3 Empirical Significance Testing

We can also assess alignment significance empirically by comparing our score against a distribution of scores from random (shuffled) sequences.

In [ ]:
import random


def shuffle_sequence(seq):
    """Return a new sequence with the same composition but shuffled order."""
    chars = list(seq)
    random.shuffle(chars)
    return "".join(chars)


def empirical_p_value(seq1, seq2, n_shuffles=999, match=2, mismatch=-1, gap=-1):
    """Estimate p-value by comparing actual score to shuffled score distribution."""
    _, _, actual_score, _ = smith_waterman(seq1, seq2, match, mismatch, gap)
    
    random_scores = []
    for _ in range(n_shuffles):
        shuffled = shuffle_sequence(seq2)
        _, _, rand_score, _ = smith_waterman(seq1, shuffled, match, mismatch, gap)
        random_scores.append(rand_score)
    
    # p-value: fraction of random scores >= actual score
    n_better = sum(1 for s in random_scores if s >= actual_score)
    p_value = (n_better + 1) / (n_shuffles + 1)  # +1 includes the actual score
    
    return actual_score, random_scores, p_value


# Test with two short related sequences
test1 = "HEAGAWGHEE"
test2 = "PAWHEAE"

actual, rand_scores, pval = empirical_p_value(test1, test2, n_shuffles=199)

print(f"Actual score: {actual}")
print(f"Random scores: mean={np.mean(rand_scores):.1f}, "
      f"max={max(rand_scores)}, min={min(rand_scores)}")
print(f"Empirical p-value: {pval:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rand_scores, bins=20, alpha=0.7, color="steelblue", edgecolor="white",
        label="Random scores")
ax.axvline(actual, color="red", linewidth=2, linestyle="--",
           label=f"Actual score = {actual}")
ax.set_xlabel("Alignment score")
ax.set_ylabel("Frequency")
ax.set_title(f"Empirical Significance (p = {pval:.4f})")
ax.legend()
plt.tight_layout()
plt.show()

---

## 9. Performance Considerations

Pairwise alignment by dynamic programming has $O(m \times n)$ time complexity. For short sequences this is fine, but for genome-scale comparisons it becomes a bottleneck.

| Sequence lengths | Matrix cells | Approximate time |
|---|---|---|
| 100 x 100 | 10,000 | < 1 ms |
| 1,000 x 1,000 | 1,000,000 | ~ 100 ms |
| 10,000 x 10,000 | 100,000,000 | ~ 10 s |
| 100,000 x 100,000 | 10,000,000,000 | ~ 17 min |

This quadratic scaling is why **heuristic methods** like BLAST were developed -- they sacrifice guaranteed optimality for dramatically faster search times, which we will cover in the next notebook.

In [ ]:
import time

# Benchmark our Python implementation
lengths = [20, 50, 100, 200]
times_list = []

for length in lengths:
    s1 = "".join(random.choices("ACGT", k=length))
    s2 = "".join(random.choices("ACGT", k=length))
    
    start = time.time()
    _ = smith_waterman(s1, s2)
    elapsed = time.time() - start
    times_list.append(elapsed)
    print(f"Length {length:>4}: {elapsed:.4f}s  ({length*length:>8,} cells)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lengths, times_list, "o-", color="steelblue")
ax.set_xlabel("Sequence length")
ax.set_ylabel("Runtime (seconds)")
ax.set_title("Smith-Waterman Runtime vs Sequence Length (pure Python)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe quadratic scaling is clearly visible.")
print("Production tools use C/SIMD implementations that are 100-1000x faster.")

---

## 10. Summary

| Concept | Key Takeaway |
|---|---|
| **Alignment goal** | Hypothesis about positional homology between sequences |
| **Dot plots** | Quick visual comparison; diagonals indicate similarity |
| **Substitution matrices** | BLOSUM62 is the default; encodes evolutionary substitution likelihoods |
| **Gap penalties** | Affine (open + extend) is more biologically realistic than linear |
| **Needleman-Wunsch** | Global alignment -- aligns sequences end-to-end |
| **Smith-Waterman** | Local alignment -- finds best matching sub-regions |
| **Percent identity** | >30% for proteins: likely homologous; 20-30%: twilight zone |
| **E-value** | Expected random hits; depends on database size |
| **Bit score** | Normalized score; database-independent |
| **Complexity** | $O(mn)$ -- motivates heuristic approaches like BLAST |

---

## Exercises

### Exercise 1: Align Two Real Proteins

Align human insulin (P01308, chain B first 30 residues) with mouse insulin (P01326) using BioPython's `PairwiseAligner` with BLOSUM62. Report the percent identity and similarity.

In [ ]:
# Exercise 1 -- Align human and mouse insulin
human_insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
mouse_insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPMS"

# YOUR CODE HERE
# 1. Create a PairwiseAligner with BLOSUM62 and affine gap penalties
# 2. Perform global alignment
# 3. Print the alignment
# 4. Calculate and print identity/similarity percentages


### Exercise 2: Compare Scoring Matrices

Align the following distantly related sequences using BLOSUM45, BLOSUM62, and BLOSUM80. Which matrix gives the best alignment for distant sequences? Why?

In [ ]:
# Exercise 2 -- Compare scoring matrices
seq_close  = "MVLSPADKTNVKAAWGKVGAHAG"  # hemoglobin alpha, human
seq_distant = "MVHLSAEEKAAVTAFWGKVKVDEVG"  # hemoglobin beta, human

# YOUR CODE HERE
# 1. Align seq_close and seq_distant with BLOSUM45, BLOSUM62, BLOSUM80
# 2. For each matrix, print the alignment and score
# 3. Which matrix gives the highest score? 
# 4. Which matrix is most appropriate for this comparison and why?


### Exercise 3: Find the Optimal Gap Penalty

Using the sequences below, systematically vary the gap open penalty from -5 to -20 (keeping extend at -0.5) and plot score vs. gap open penalty. At what penalty does the alignment change qualitatively?

In [ ]:
# Exercise 3 -- Gap penalty optimization
seq_x = "MTEYKLVVVGAVGVGKSALTIQLIQNHFVDEYDPTIEDSY"
seq_y = "MTEYKLVVLGAVGIGKSALTIQFIQSHFVDQYDPTIEDSY"

# YOUR CODE HERE
# 1. Loop over gap_open values: -5, -6, -7, ..., -20
# 2. For each, align with BLOSUM62 and gap_extend = -0.5
# 3. Record the score and number of gaps
# 4. Plot: score vs gap_open and gaps vs gap_open


### Exercise 4: Implement a Dot Plot Filter

Modify the `windowed_dot_plot` function to accept a substitution matrix instead of exact matching. A dot should be placed when the average substitution score within the window exceeds a threshold.

In [ ]:
# Exercise 4 -- Substitution-matrix-aware dot plot

# YOUR CODE HERE
# 1. Define a function scored_dot_plot(seq1, seq2, sub_matrix, window, threshold)
# 2. For each window position, compute the average substitution score
# 3. Place a dot if the average exceeds the threshold
# 4. Test on the hemoglobin alpha (human vs mouse) sequences


### Exercise 5: Global vs Local -- When Does It Matter?

A researcher has a short domain sequence (30 aa) and wants to find it within a larger protein (200 aa). 

1. Generate a random protein of 200 amino acids
2. Insert your domain at position 80-110
3. Align the domain against the full protein using both global and local alignment
4. Which gives a better result? Why?

In [ ]:
# Exercise 5 -- Global vs Local alignment
domain = "HEAGAWGHEEPAWHEAEQWRTYI"

# YOUR CODE HERE
# 1. Generate a random 200 aa protein (use random.choices with standard amino acids)
# 2. Insert the domain at position 80
# 3. Align domain vs full protein with BOTH global and local alignment
# 4. Print both alignments and scores
# 5. Explain which is better and why


### Exercise 6: Assess Homology

Write a function `assess_homology(e_value, identity_pct, alignment_length)` that returns a verdict about whether two sequences are likely homologous. Test it with the values in the table below.

| E-value | Identity | Length | Expected verdict |
|---|---|---|---|
| 1e-80 | 95% | 300 | Certain homologs |
| 1e-20 | 45% | 150 | Likely homologs |
| 0.05 | 22% | 80 | Twilight zone |
| 5.0 | 15% | 40 | Not significant |

In [ ]:
# Exercise 6 -- Homology assessment function

# YOUR CODE HERE
# 1. Define assess_homology(e_value, identity_pct, alignment_length)
# 2. Consider E-value thresholds, identity zones, and alignment length
# 3. Return a tuple of (verdict_string, confidence_score)
# 4. Test with all four cases above


---

## Further Reading

- Needleman SB, Wunsch CD (1970). A general method applicable to the search for similarities in the amino acid sequence of two proteins. *J Mol Biol* 48:443-453.
- Smith TF, Waterman MS (1981). Identification of common molecular subsequences. *J Mol Biol* 147:195-197.
- Henikoff S, Henikoff JG (1992). Amino acid substitution matrices from protein blocks. *PNAS* 89:10915-10919.
- Dayhoff MO, Schwartz RM, Orcutt BC (1978). A model of evolutionary change in proteins. *Atlas of Protein Sequence and Structure* 5:345-352.
- Karlin S, Altschul SF (1990). Methods for assessing the statistical significance of molecular sequence features. *PNAS* 87:2264-2268.
- BioPython Tutorial: [Pairwise Alignment](https://biopython.org/docs/latest/Tutorial/chapter_pairwise.html)

**Next notebook**: BLAST -- heuristic sequence database searching at scale.

---

[← Previous: BioPython Essentials](../../Tier_2_Core_Bioinformatics/02_BioPython_Essentials/01_biopython_essentials.ipynb) | [Next: BLAST: Sequence Similarity Searching →](../../Tier_2_Core_Bioinformatics/04_BLAST_Searching/01_blast_searching.ipynb)